In [42]:
import os
import sys

# add the source directory to system path, so that relative imports work
# this fix is only for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pykep

from scipy.optimize import least_squares
from orbital_mechanics.solar_system import SolarSystem
from common.constants import ALTAIRA_MU as MU, ALTAIRA_AU as AU, YEAR

In [44]:
ss = SolarSystem()
planets = ss.bodies[ss.planets_idx]

In [45]:
planets[0]

Id: 1
Type: planet
Weight: 0.1
Planet Name: Vulcan
Own gravity parameter: 658906373.32000005
Central body gravity parameter: 139348062043.34299
Planet radius: 133020.70000000001
Planet safe radius: 146322.77000000002
Keplerian planet elements: 
Semi major axis (AU): 9.2327403312639171e-05
Eccentricity: 0
Inclination (deg.): 0
Big Omega (deg.): 0
Small omega (deg.): 315.37200000000001
Mean anomaly (deg.): 322.584
Elements reference epoch: 2000-Jan-01 00:00:00
Ephemerides type: Keplerian
r at ref. = [1911752.3142579421, -13679037.827238742, 0]
v at ref. = [99.476836844647067, 13.902664460370346, 0]

In [46]:
sequence = np.array(sorted(planets, key=lambda p: p.osculating_elements(pykep.epoch(0))[0], reverse=True))

print("Initial sequence.. inner planets from jotunn onwards may or may not be visited")
print([p.name for p in sequence])
print(f'{len(sequence)} planets')

Initial sequence.. inner planets from jotunn onwards may or may not be visited
['PlanetX', 'Rogue1', 'Wakonyingo', 'Jotunn', 'Bespin', 'Beyonc', 'Yandi', 'Hoth', 'Eden', 'Yavin', 'Vulcan']
11 planets


r0x = -29919574138.2  # km
r0y = 3893598350.55096  # km
r0z = -2257832147.745   # km

r0x = -29919574138  # km
ry0 = -2270724.609  # km
rz0 = -5648624511   # km

In [57]:
class InitialSequenceProblem:
    def __init__(self, mu, r0, planet_seq, tf, var_bounds):
        self.mu = mu
        self.r0 = r0
        self.planet_seq = planet_seq.copy()   # here planet_seq is an array of pykep planets
        self.num_planets = len(self.planet_seq)
        self.tf = tf
        self.var_bounds = var_bounds

    def fitness(self, tof_arr):
        # x -> time of flights between the planets
        # there should be len(planet_seq) tof numbers
        assert len(tof_arr) == self.num_planets

        # cumulative time array
        t_arr = np.cumsum(tof_arr)

        # construct the arguments for the lambert problems
        r1_arr = np.empty((self.num_planets,3), dtype=np.float64)
        r2_arr = np.empty((self.num_planets,3), dtype=np.float64)
        cw_arr = np.empty(self.num_planets, dtype=np.bool)

        for i in range(self.num_planets):
            plnt = self.planet_seq[i]
            r,_ = plnt.eph(t_arr[i]*pykep.SEC2DAY)
            r2_arr[i] = np.array(r)

        r1_arr[1:] = r2_arr[:-1]
        r1_arr[0] = r0

        for i in range(self.num_planets):
            cw_arr[i] = np.cross(r1_arr[i], r2_arr[i])[2] < 0

        # solve the lambert problems
        lambert_sols = np.empty(self.num_planets, dtype=object)

        for i in range(self.num_planets):
            lambert_sols[i] = pykep.lambert_problem(r1_arr[i], r2_arr[i],
                                            tof_arr[i], self.mu, int(cw_arr[i]), 0)

        
        vin = np.empty((self.num_planets-1,3), dtype=np.float64)
        vout = np.empty((self.num_planets-1,3), dtype=np.float64)
        
        for i in range(self.num_planets-1):
            plnt = self.planet_seq[i]
            _,vpl = plnt.eph(t_arr[i]*pykep.DAY2SEC)
            vpl = np.array(vpl)
            vsc_in = np.array(lambert_sols[i].get_v2()[0])
            vsc_out = np.array(lambert_sols[i+1].get_v1()[0])
            vin[i] = vsc_in-vpl
            vout[i] = vsc_out-vpl

        veq_const = np.empty(self.num_planets-1, dtype=np.float64)
        vineq_const = np.empty(self.num_planets-1, dtype=np.float64)

        for i in range(self.num_planets-1):
            eq,ineq = pykep.fb_con(vin[i], vout[i], self.planet_seq[i])
            veq_const[i] = eq
            vineq_const[i] = ineq
        
        veq_const = np.nan_to_num(veq_const)
        vineq_const = np.nan_to_num(vineq_const)

        # get initial velocity
        v0 = np.array(lambert_sols[0].get_v1()[0])

        # obj = t_arr[-1]  # fastest time
        obj = np.concatenate([v0[1:], veq_const]) #vineq_const])

        # -> [obj, equality_const, inequality_const]
        
        obj = np.linalg.norm(obj)
        print(obj)

        return obj

    
    def get_bounds(self):
        # -> bounds for variables
        return self.var_bounds

In [58]:
r0 = np.array([-29919574138.2, 3893598350.55096, -2257832147.745])
r0 = np.array([-29919574138, -2270724.609, -5648624511])

var_min_bounds = np.array([5, 10, 10, 5, 3, 1.2, 0.5, 0.2, 0.1, 0.1, 0.1]) * YEAR
var_max_bounds = np.array([150, 100, 100, 50, 30, 12, 5, 2, 1, 1, 1.]) * YEAR
var_bounds = (var_min_bounds, var_max_bounds)

prob = InitialSequenceProblem(MU, r0, sequence, 200*YEAR, var_bounds)
prob

In [61]:
al = np.random.uniform(low=0, high=1, size=11)
x0 = var_min_bounds + (var_max_bounds-var_min_bounds)*al

res = least_squares(lambda x: prob.fitness(x), x0, bounds=prob.get_bounds())
print(res['x']/YEAR)
print(prob.fitness(res['x']))

4865.099637833307
4683.6547209705295
4554.719084953721
4215.665055581476
3645.623001985438
3399.10919668375
5368.613019012146
5387.99495637809
5005.608242575215
3649.4192705285222
5978.003745464036
4865.09980069041
9724.240965775276
2741.602254991042
5365.4039743073945
4901.15566819834
4163.488622728928
5188.8074308695095
5626.089852098828
4771.86191010559
4086.33461075923
4022.1492444217697
5343.655695630145
4317.430766848811
2741.602195030647
7093.379578139034
4733.4362824982045
3683.0610078821164
5790.055138457552
2421.1040687936625
7778.507011194205
7543.24969906281
4181.660003239229
4158.972380104021
4841.302710490131
2481.0792206969413
2830.703340324958
4393.647364495203
5179.947651120694
2464.9037326540824
2421.1038466654354
4826.857147725665
5576.616392351613
4004.888670595724
6246.296512427741
4097.165852709049
5615.545217036111
6000.750352958855
5704.712106248541
[89.47355906 61.54057345 37.60850521 15.11869841 21.51372044  5.66235119
  2.73825963  0.54161466  0.98316766  0.6

In [62]:
res['x'] / YEAR

array([89.47355906, 61.54057345, 37.60850521, 15.11869841, 21.51372044,
        5.66235119,  2.73825963,  0.54161466,  0.98316766,  0.60331753,
        0.93963487])

In [53]:
prob.fitness(res['x'])

array([ 5.76935381e-01,  4.89522320e-01, -3.27932236e+01,  2.15428793e+01,
       -3.75721209e+01, -1.50374755e+02,  1.68560987e+02,  3.48280005e+02,
       -4.44873926e+02, -1.56139018e+03,  5.75182104e+02,  8.69972476e+02])